# Digital Image Processing Lab - Assignment 9
**Problem Statements:**
1. **Write a program to implement the Local Binary Pattern (LBP) on gray scale image.**
2. **Write a program to denoise the input image corrupted by Gaussian and Speckle noise by using following filters ($3\times 3$ or $5\times 5$ window):**
   - (a) **Wiener filter**
   - (b) **Median filtering**
   - (c) **Gaussian filter**
   - (d) **Bilateral filter**

---
### Theoretical Foundations
1. **Local Binary Pattern (LBP):**
   A powerful texture descriptor comparing central pixel $g_c$ with $P=8$ surrounding neighbors $g_p$:
   $$\text{LBP}(x_c, y_c) = \sum_{p=0}^{7} s(g_p - g_c) \cdot 2^p, \quad s(x) = \begin{cases} 1 & x \ge 0 \\ 0 & x < 0 \end{cases}$$

2. **Denoising Filters:**
   - **Gaussian Noise:** Additive zero-mean noise $I_{\text{noisy}} = I + n$.
   - **Speckle Noise:** Multiplicative noise $I_{\text{noisy}} = I + I \cdot n$.
   - **Filters:**
     - **Wiener:** Optimal minimum mean-square error adaptive filter: $\hat{f} = \mu + \frac{\sigma^2 - \nu^2}{\sigma^2}(g - \mu)$
     - **Median:** Non-linear rank-order filter, replaces pixel with neighborhood median.
     - **Gaussian:** Linear smoothing filter using 2D normal distribution.
     - **Bilateral:** Edge-preserving smoothing combining spatial closeness and photometric range similarity.


In [ ]:
# Cell 1: Import Libraries
import numpy as np
import matplotlib.pyplot as plt
import cv2
from scipy.signal import wiener

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['image.cmap'] = 'gray'


In [ ]:
# Cell 2: Test Image Generator
def get_texture_image():
    # Image with both smooth regions, edges, and high-frequency textures
    x = np.linspace(-4, 4, 256)
    y = np.linspace(-4, 4, 256)
    xx, yy = np.meshgrid(x, y)
    texture = np.sin(xx * 5) * np.cos(yy * 5) * 60 + 128
    
    cv2.circle(texture, (80, 80), 45, 230, -1)
    cv2.rectangle(texture, (130, 130), (210, 210), 30, -1)
    return np.clip(texture, 0, 255).astype(np.uint8)

clean_img = get_texture_image()
print("Reference image generated.")


In [ ]:
# Cell 3: Task 1 - Local Binary Pattern (LBP) From Scratch
def compute_lbp(image):
    """Computes 8-neighbor Local Binary Pattern for each pixel from scratch."""
    h, w = image.shape
    lbp_image = np.zeros((h - 2, w - 2), dtype=np.uint8)
    
    # 8-neighbor relative coordinates (clockwise)
    offsets = [
        (-1, -1), (-1, 0), (-1, 1),
        (0, 1),
        (1, 1), (1, 0), (1, -1),
        (0, -1)
    ]
    
    center = image[1:-1, 1:-1]
    
    for power, (dr, dc) in enumerate(offsets):
        neighbor = image[1+dr:h-1+dr, 1+dc:w-1+dc]
        # s(neighbor - center) >= 0
        bit = (neighbor >= center).astype(np.uint8)
        lbp_image += bit * (1 << power)
        
    return lbp_image

lbp_result = compute_lbp(clean_img)
print(f"LBP map computed. Shape: {lbp_result.shape}, Range: [{lbp_result.min()}, {lbp_result.max()}]")


In [ ]:
# Cell 4: Task 2 - Noise Simulation (Gaussian & Speckle)
np.random.seed(42)

# 1. Additive Gaussian noise
gauss_noise = np.random.normal(0, 25, clean_img.shape)
img_gauss = np.clip(clean_img.astype(np.float64) + gauss_noise, 0, 255).astype(np.uint8)

# 2. Multiplicative Speckle noise: I + I * n
speckle_noise = np.random.normal(0, 0.2, clean_img.shape)
img_speckle = np.clip(clean_img.astype(np.float64) + clean_img.astype(np.float64) * speckle_noise, 0, 255).astype(np.uint8)

print("Gaussian and Speckle noisy images synthesized.")


In [ ]:
# Cell 5: Task 2 - Application of 4 Denoising Filters (3x3 or 5x5)
filter_size = 5 #@param [3, 5] {type:"raw"}

def apply_filters(noisy_img, ksize):
    """Applies Wiener, Median, Gaussian, and Bilateral filters."""
    # (a) Wiener Filter
    denoised_wiener = wiener(noisy_img.astype(np.float64), mysize=(ksize, ksize))
    denoised_wiener = np.clip(denoised_wiener, 0, 255).astype(np.uint8)
    
    # (b) Median Filter
    denoised_median = cv2.medianBlur(noisy_img, ksize)
    
    # (c) Gaussian Filter
    denoised_gaussian = cv2.GaussianBlur(noisy_img, (ksize, ksize), sigmaX=1.5)
    
    # (d) Bilateral Filter
    denoised_bilateral = cv2.bilateralFilter(noisy_img, d=ksize, sigmaColor=75, sigmaSpace=75)
    
    return {
        "Wiener": denoised_wiener,
        "Median": denoised_median,
        "Gaussian": denoised_gaussian,
        "Bilateral": denoised_bilateral
    }

results_gauss = apply_filters(img_gauss, filter_size)
results_speckle = apply_filters(img_speckle, filter_size)

def calculate_psnr(orig, filtered):
    mse = np.mean((orig.astype(float) - filtered.astype(float)) ** 2)
    return 10 * np.log10(255.0**2 / mse) if mse > 0 else float('inf')

print(f"--- PSNR Comparison for Filter Window {filter_size}x{filter_size} ---")
print(f"Initial Noisy: Gaussian = {calculate_psnr(clean_img, img_gauss):.2f} dB, Speckle = {calculate_psnr(clean_img, img_speckle):.2f} dB")
for name in ["Wiener", "Median", "Gaussian", "Bilateral"]:
    psnr_g = calculate_psnr(clean_img, results_gauss[name])
    psnr_s = calculate_psnr(clean_img, results_speckle[name])
    print(f"{name:10s} -> Gaussian: {psnr_g:.2f} dB | Speckle: {psnr_s:.2f} dB")


In [ ]:
# Cell 6: Visualizations
# Plot Task 1: LBP
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(clean_img); axes[0].set_title("Original Image", fontweight='bold'); axes[0].axis('off')
axes[1].imshow(lbp_result, cmap='gray'); axes[1].set_title("Local Binary Pattern (LBP) Texture Map", fontweight='bold'); axes[1].axis('off')
axes[2].hist(lbp_result.ravel(), bins=256, range=[0, 256], color='navy', alpha=0.7)
axes[2].set_title("LBP Feature Histogram (256 bins)", fontweight='bold'); axes[2].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Plot Task 2: Denoising Matrices
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
# Gaussian Row
axes[0, 0].imshow(img_gauss); axes[0, 0].set_title("Noisy: Gaussian", fontweight='bold'); axes[0, 0].axis('off')
axes[0, 1].imshow(results_gauss['Wiener']); axes[0, 1].set_title("Wiener", fontweight='bold'); axes[0, 1].axis('off')
axes[0, 2].imshow(results_gauss['Median']); axes[0, 2].set_title("Median", fontweight='bold'); axes[0, 2].axis('off')
axes[0, 3].imshow(results_gauss['Gaussian']); axes[0, 3].set_title("Gaussian", fontweight='bold'); axes[0, 3].axis('off')
axes[0, 4].imshow(results_gauss['Bilateral']); axes[0, 4].set_title("Bilateral", fontweight='bold'); axes[0, 4].axis('off')

# Speckle Row
axes[1, 0].imshow(img_speckle); axes[1, 0].set_title("Noisy: Speckle", fontweight='bold'); axes[1, 0].axis('off')
axes[1, 1].imshow(results_speckle['Wiener']); axes[1, 1].set_title("Wiener", fontweight='bold'); axes[1, 1].axis('off')
axes[1, 2].imshow(results_speckle['Median']); axes[1, 2].set_title("Median", fontweight='bold'); axes[1, 2].axis('off')
axes[1, 3].imshow(results_speckle['Gaussian']); axes[1, 3].set_title("Gaussian", fontweight='bold'); axes[1, 3].axis('off')
axes[1, 4].imshow(results_speckle['Bilateral']); axes[1, 4].set_title("Bilateral", fontweight='bold'); axes[1, 4].axis('off')

plt.tight_layout()
plt.show()


### Conclusions
- **LBP:** Highly discriminative, illumination-invariant descriptor capturing microscopic surface textures.
- **Wiener Filter:** Statistically adapts to local variance, yielding exceptional PSNR for additive Gaussian noise.
- **Bilateral Filter:** Preserves sharp edges while smoothing uniform noisy backgrounds.
